# bioRxiv Tools

![bioRxiv Tools](https://proto-bio.github.io/proto-assets/images/tool/biorxiv/hero.png)

This notebook covers `run_biorxiv_details`, which looks up bioRxiv and medRxiv preprints by DOI. It returns every posted version with its date, the latest title, authors, category, license, and abstract, and the journal DOI once the preprint has been published. The bioRxiv API has no keyword search, so this notebook also shows how to find preprint DOIs with `run_europepmc_search`.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("biorxiv")
display_overview("biorxiv")
display_docs_section("biorxiv", "Background")

# bioRxiv

[bioRxiv](https://www.biorxiv.org/) and [medRxiv](https://www.medrxiv.org/) are preprint servers for the life sciences and health sciences, run by openRxiv, an independent non-profit. This toolkit exposes one tool, `biorxiv-details`, which looks up a preprint by DOI through the bioRxiv API and returns every posted version with its date, the title, authors, category, license, abstract, and the journal DOI once the preprint is published. It runs on CPU in process and needs only network access.

[bioRxiv](https://www.biorxiv.org/) opened in 2013 so that researchers could share manuscripts before peer review, and [medRxiv](https://www.medrxiv.org/) followed in 2019 for clinical and health research. Preprints are screened but not peer reviewed. Authors can post revised versions, so one DOI may carry several versions, each with its own date and text, and a figure or value can change between them. When a preprint is later published, the servers link it to the journal article's DOI.

The [bioRxiv API](https://api.biorxiv.org/) serves this metadata for both servers. It looks records up by DOI or lists them by posting date, but it has no keyword search.

## Available tools

In [2]:
display_available_tools("biorxiv")

- **`run_biorxiv_details()`** — Look up bioRxiv or medRxiv preprint metadata by DOI: versions and dates, authors, abstract, and the published journal DOI

### `run_biorxiv_details`

`run_biorxiv_details` takes one or more DOIs and returns one record per DOI. Here we look up the RFdiffusion preprint, which was revised once and later published in Nature.

In [3]:
from proto_tools import (
    BiorxivDetailsConfig,
    BiorxivDetailsInput,
    run_biorxiv_details,
)

In [4]:
# Display docs
display_api_reference("biorxiv", "input", "run_biorxiv_details")

# Look up the RFdiffusion preprint; a doi.org link is accepted
inputs = BiorxivDetailsInput(dois=["https://doi.org/10.1101/2022.12.09.519842"])

**Input** — `BiorxivDetailsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>dois</code> | <code>list[str]</code> | required | bioRxiv or medRxiv DOIs, e.g. '10.1101/2022.12.09.519842' |

In [5]:
# Display config docs
display_api_reference("biorxiv", "config", "run_biorxiv_details")

# Try bioRxiv, then medRxiv (the default) | see docs above for all fields
config = BiorxivDetailsConfig(server="auto")

**Config** — `BiorxivDetailsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>server</code> | <code>Literal['auto', 'biorxiv', 'medrxiv']</code> | <code>'auto'</code> | 'auto' tries bioRxiv then medRxiv; 'biorxiv' or 'medrxiv' queries only that one |

In [6]:
# Run the lookup
output = run_biorxiv_details(inputs, config)

In [7]:
# Display output docs
display_api_reference("biorxiv", "output", "run_biorxiv_details")

# Inspect the record and its versions
preprint = output.preprints[0]
print(f"{preprint.title}\n{preprint.server}  {preprint.category}  license={preprint.license}")
print(f"Authors: {', '.join(preprint.authors[:5])}, ... ({len(preprint.authors)} total)")
print(f"Published as: {preprint.published_doi}")
for version in preprint.versions:
    print(f"  v{version.version}  {version.date}  {version.jats_xml_url}")

**Output** — `BiorxivDetailsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>preprints</code> | <code>list[BiorxivPreprint]</code> | <code>[]</code> | Records in input order |
| <code>source_url</code> | <code>str</code> | required | bioRxiv API endpoint used |

**`BiorxivPreprint`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>doi</code> | <code>str</code> | required | Requested preprint DOI |
| <code>error</code> | <code>str &#124; None</code> | <code>None</code> | Why this item has no data; None when it was retrieved |
| <code>server</code> | <code>str &#124; None</code> | <code>None</code> | Server that answered, as reported |
| <code>title</code> | <code>str &#124; None</code> | <code>None</code> | Title of the latest version |
| <code>authors</code> | <code>list[str]</code> | <code>[]</code> | Authors of the latest version |
| <code>corresponding_author</code> | <code>str &#124; None</code> | <code>None</code> | Corresponding author |
| <code>corresponding_institution</code> | <code>str &#124; None</code> | <code>None</code> | Corresponding author's institution |
| <code>category</code> | <code>str &#124; None</code> | <code>None</code> | Subject category |
| <code>license</code> | <code>str &#124; None</code> | <code>None</code> | License code, e.g. 'cc_by' |
| <code>abstract</code> | <code>str &#124; None</code> | <code>None</code> | Abstract of the latest version |
| <code>published_doi</code> | <code>str &#124; None</code> | <code>None</code> | Journal DOI once published, else None |
| <code>versions</code> | <code>list[BiorxivVersion]</code> | <code>[]</code> | All posted versions, oldest first |
| <code>source_url</code> | <code>str &#124; None</code> | <code>None</code> | bioRxiv API URL the record was read from |

**`BiorxivVersion`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>version</code> | <code>int</code> | required | Version number |
| <code>date</code> | <code>str</code> | required | Posting date (YYYY-MM-DD) |
| <code>type</code> | <code>str &#124; None</code> | <code>None</code> | Article type, e.g. 'new results' |
| <code>jats_xml_url</code> | <code>str &#124; None</code> | <code>None</code> | This version's JATS XML |

Broadly applicable and accurate protein design by integrating structure prediction networks and diffusion generative models
bioRxiv  biochemistry  license=cc_by_nd
Authors: Watson, J. L., Juergens, D., Bennett, N. R., Trippe, B. L., Yim, J., ... (27 total)
Published as: 10.1038/s41586-023-06415-8
  v1  2022-12-10  https://www.biorxiv.org/content/early/2022/12/10/2022.12.09.519842.source.xml
  v2  2022-12-14  https://www.biorxiv.org/content/early/2022/12/14/2022.12.09.519842.source.xml


#### From a Keyword Search to Preprint Details

To find preprints by topic, search Europe PMC with `SRC:PPR` and pass the DOIs of the hits to `run_biorxiv_details`. Here we take the most recent bioRxiv hits for protein binder design and check their version history and publication status.

In [8]:
from proto_tools import EuropePMCSearchConfig, EuropePMCSearchInput, run_europepmc_search

hits = run_europepmc_search(
    EuropePMCSearchInput(queries='TITLE:"binder design" AND SRC:PPR AND PUBLISHER:"bioRxiv"'),
    EuropePMCSearchConfig(max_results=3, include_abstracts=False),
).results[0].hits

details = run_biorxiv_details(BiorxivDetailsInput(dois=[hit.doi for hit in hits if hit.doi]))
for preprint in details.preprints:
    latest = preprint.versions[-1]
    print(f"{preprint.doi}  v{latest.version} ({latest.date})  published={preprint.published_doi}  {preprint.title[:60]}")

10.64898/2026.09.30.755438  v1 (2026-10-01)  published=None  pH-sensitive binder design with Proton-PottsMPNN
10.64898/2026.10.22.760942  v1 (2026-10-23)  published=None  Projected Gradient Descent for Hallucination-Based Protein B
10.64898/2026.10.29.762730  v1 (2026-10-30)  published=None  LPT: Orchestrating bounded agentic loops for automated prote


## Export Results

Records can be saved to JSON for downstream use. Preprint text and abstracts keep their authors' licenses, so this example writes to a temporary directory.

In [9]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as output_dir:
    output.export("biorxiv_details", export_path=output_dir, file_format="json")
    print(f"Exported {sorted(path.name for path in Path(output_dir).iterdir())}")

Exported ['biorxiv_details.json']
